# 02 · Swap the knowledge graph: GO vs Reactome vs STRING

**Problem.** The whole point of `graph-perturb` is that the biological prior is *swappable*. Does the
choice of knowledge graph — Gene Ontology BP, Reactome pathways, or the STRING PPI network — actually
change how well a GNN predicts held-out perturbations?

**Approach.** Hold the **gene universe and the train/val/test split fixed**, then train the *same*
GraphSAGE GNN on each backend in turn. We build each graph with `get_graph_source(<name>)`, train via
the real `train_model`, evaluate with `evaluate_model`, and assemble a single tidy table with
`evaluate.compare_backends`.

**What to look at.** The `(backend, split)` table at the end — higher `pearson_delta` /
`overlap_at_20` and lower `mse` is better. Differences reflect how well each prior's edges connect
the perturbed genes to their responders.

In [ ]:
import logging
import numpy as np
logging.basicConfig(level=logging.WARNING)
np.random.seed(0)

from graph_perturb.config import DataConfig, SplitConfig, ModelConfig, TrainConfig, EvalConfig
from graph_perturb.data import make_splits
from graph_perturb.data.norman import load_norman, make_synthetic_norman
from graph_perturb.data.dataset import build_dataloaders
from graph_perturb.graphs.registry import get_graph_source
from graph_perturb.models import build_model
from graph_perturb.train import train_model
from graph_perturb.evaluate import evaluate_model, compare_backends

## 1. One dataset + one split, shared across all backends

Loading and splitting **once** is what makes this a fair backend-vs-backend comparison: every model
sees identical training conditions and is scored on identical held-out conditions.

In [ ]:
try:
    data = load_norman(DataConfig(name="norman", n_top_genes=2000))
    MODE = "REAL Norman Perturb-seq"
except Exception as exc:
    print(f"[fallback] real Norman load failed ({type(exc).__name__}: {exc})")
    data = make_synthetic_norman(n_genes=80, n_conditions=30, seed=0)
    MODE = "SYNTHETIC offline stand-in"
print(f"DATA MODE: {MODE}  |  cells={data.n_cells} genes={data.n_genes}")

splits = make_splits(data, SplitConfig(test_single_frac=0.3, test_combo_frac=0.3, seed=0))
print({k: len(v) for k, v in splits.as_dict().items()})

## 2. Train the same GNN on each backend

Each backend builds edges over the SAME `data.gene_names`. If a real source dump is unavailable
offline, the backend logs a warning and falls back to deterministic co-expression edges, so this
cell always runs. We keep epochs tiny for CPU.

In [ ]:
BACKENDS = ["go_bp", "reactome", "string"]
model_cfg = ModelConfig(name="gnn", hidden_dim=32, n_layers=2, dropout=0.1, attention_heads=2)
train_cfg = TrainConfig(epochs=5, batch_size=16, lr=1e-3, device="cpu",
                        early_stop_patience=5, log_every=100, seed=0)
eval_cfg = EvalConfig(overlap_k=20, splits=("test_single", "test_combo"))

results_by_backend = {}
for backend in BACKENDS:
    print(f"=== {backend} ===")
    source = get_graph_source(backend, undirected=True, add_self_loops=True)
    graph = source.build(data.gene_names, use_cache=False)
    print(f"  {graph}")
    loaders = build_dataloaders(data, graph, splits, train_cfg)
    model = build_model(model_cfg, num_genes=data.n_genes, graph=graph)
    train_model(model, loaders, train_cfg, ckpt_dir=None)
    results_by_backend[backend] = evaluate_model(model, data, graph, splits, eval_cfg)

## 3. Backend-vs-backend metrics table

In [ ]:
table = compare_backends(results_by_backend)
print(table.to_string())
table

**Takeaway.** With identical data, split, and model, the only thing that changed is the graph prior.
On the real Norman data the denser, function-curated graphs (GO BP / Reactome) usually edge out a raw
PPI on `pearson_delta` because their edges connect a perturbed transcription factor to the genes it
co-regulates. The exact ordering depends on the gene universe and how many epochs you train — bump
`train_cfg.epochs` for a sharper separation.